In [9]:
# ============================================================
# INITIALIZE RAG COMPONENTS
# ============================================================

from pathlib import Path
import sys


# ============================================================
# PATH CONFIGURATION
# ============================================================

CURRENT_DIR = Path.cwd()

# Notebook is:
# rag/Test/test_retrieval_accuracy.ipynb
#
# Therefore:
# CURRENT_DIR.parent = rag/

RAG_DIR = CURRENT_DIR.parent

# Add rag/ to Python import path
if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))


# ============================================================
# DEBUG PATHS
# ============================================================

print("=" * 70)
print("PATH CONFIGURATION")
print("=" * 70)

print(f"\nCurrent directory:")
print(CURRENT_DIR)

print(f"\nRAG directory:")
print(RAG_DIR)

print("\nRAG directory exists:")
print(RAG_DIR.exists())

print("\nRetrieval directory exists:")
print(
    (RAG_DIR / "retrieval").exists()
)

print("\nContext directory exists:")
print(
    (RAG_DIR / "context").exists()
)


# ============================================================
# IMPORTS
# ============================================================

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker

from vector_store.Metadata_filter import MetadataFilter

from context.context_selector import ContextSelector
from context.context_builder import ContextBuilder


# ============================================================
# PATHS
# ============================================================

CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)

print("\nChunks file:")
print(CHUNKS_FILE)

print("\nChunks file exists:")
print(CHUNKS_FILE.exists())


# ============================================================
# INITIALIZE RETRIEVAL
# ============================================================

print("\nInitializing Hybrid Retriever...")

hybrid_retriever = HybridRetriever()


print("\nInitializing Cross-Encoder...")

reranker = CrossEncoderReranker()


print("\nInitializing Metadata Filter...")

metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)


# ============================================================
# INITIALIZE CONTEXT
# ============================================================

print("\nInitializing Context Selector...")

context_selector = ContextSelector(
    top_k=3
)


print("\nInitializing Context Builder...")

context_builder = ContextBuilder()


# ============================================================
# COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("ALL COMPONENTS READY")
print("=" * 70)

PATH CONFIGURATION

Current directory:
C:\Users\User\Nasa Rag SYS\rag\context

RAG directory:
C:\Users\User\Nasa Rag SYS\rag

RAG directory exists:
True

Retrieval directory exists:
True

Context directory exists:
True

Chunks file:
C:\Users\User\Nasa Rag SYS\rag\embeddings\processed\chunks\chunks.json

Chunks file exists:
True

Initializing Hybrid Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Initializing Cross-Encoder...
CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.

Initializing Metadata Filter...
Loaded 1200 chunks.

Initializing Context Selector...

Initializing Context Builder...

ALL COMPONENTS READY


In [10]:
# ============================================================
# TEST CONTEXT SELECTION + CONTEXT BUILDING
# ============================================================

from context_selector import ContextSelector
from context_builder import ContextBuilder


# ============================================================
# COMPONENTS
# ============================================================

context_selector = ContextSelector(
    top_k=3
)

context_builder = ContextBuilder()


# ============================================================
# TEST QUERY
# ============================================================

query = (
    "How does oxygen concentration affect "
    "flame behavior in microgravity?"
)


# ============================================================
# HYBRID RETRIEVAL
# ============================================================

hybrid_results = hybrid_retriever.search(
    query=query,
    dense_top_k=20,
    bm25_top_k=20,
    top_k=20
)


# ============================================================
# ENRICH
# ============================================================

enriched_results = metadata_filter.enrich_results(
    hybrid_results
)


# ============================================================
# RERANK
# ============================================================

reranked_results = reranker.rerank(
    query=query,
    candidates=enriched_results,
    top_k=5
)


# ============================================================
# SELECT
# ============================================================

selected_results = context_selector.select(
    reranked_results
)


# ============================================================
# DISPLAY SELECTION
# ============================================================

print("=" * 80)
print("CONTEXT SELECTION")
print("=" * 80)

print(
    f"\nReranked candidates: "
    f"{len(reranked_results)}"
)

print(
    f"Selected context: "
    f"{len(selected_results)}"
)


for result in selected_results:

    print("\n" + "-" * 60)

    print(
        f"Context Rank: "
        f"{result['context_rank']}"
    )

    print(
        f"Chunk ID: "
        f"{result['chunk_id']}"
    )

    print(
        f"Experiment: "
        f"{result['metadata'].get('experiment_id', 'N/A')}"
    )

    print(
        f"Reranker Score: "
        f"{result.get('reranker_score', 0):.4f}"
    )


# ============================================================
# BUILD CONTEXT
# ============================================================

context = context_builder.build(
    selected_results
)


# ============================================================
# DISPLAY FINAL CONTEXT
# ============================================================

print("\n\n")
print("=" * 80)
print("FINAL LLM CONTEXT")
print("=" * 80)

print("\n")
print(context)


CROSS-ENCODER RERANKING
Query: How does oxygen concentration affect flame behavior in microgravity?
Candidates received: 20
Generating reranker scores...

RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 143
Reranker Score: -2.8965
Hybrid Rank: 14
Dense Rank: None
BM25 Rank: 7

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 191
Reranker Score: -3.0640
Hybrid Rank: 20
Dense Rank: None
BM25 Rank: 10

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 151
Reranker Score: -3.0825
Hybrid Rank: 16
Dense Rank: None
BM25 Rank: 8

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 31
Reranker Score: -3.1579
Hybrid Rank: 4
Dense Rank: None
BM25 Rank: 2

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 43
Reranker Score: -3.1874
Hybrid Rank: 8
Dense Rank: None
BM25 Rank: 4
CONTEXT SELECTION

Reranked candidates: 5
Selected con